# 04.1 — Build Master Races from Ergast (2018+) + Kaggle History

Rebuild `master_races.csv` using:
- **Kaggle** tables for 1994–2017 (unchanged schema)
- **Ergast/Jolpica** exports in `data/raw/ergast/` for 2018–latest

Ergast per-year files are normalized to the exact Kaggle table shapes (`results`, `races`, `qualifying`, standings, sprint) before the same merge logic as `02_data_combining.ipynb`.

**Output:** `data/processed/master_races_ergast.csv`

In [3]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ergast_kaggle_adapter import (
    ERGAST_START_YEAR,
    ERGAST_YEARS_DEFAULT,
    MIN_MASTER_YEAR,
    merge_historical_and_ergast_kaggle_tables,
)
from master_combine import build_master_races

KAGGLE_ROOT = PROJECT_ROOT / "data" / "raw" / "kaggle"
ERGAST_ROOT = PROJECT_ROOT / "data" / "raw" / "ergast"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_ROOT / "master_races_ergast.csv"
ERGAST_YEARS = range(ERGAST_START_YEAR, 2026)

print(f"Kaggle ref: {KAGGLE_ROOT}")
print(f"Ergast:     {ERGAST_ROOT}")
print(f"Years:      {MIN_MASTER_YEAR}–{max(ERGAST_YEARS)} (Ergast from {ERGAST_START_YEAR})")

Kaggle ref: C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\raw\kaggle
Ergast:     C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\raw\ergast
Years:      1994–2025 (Ergast from 2018)


In [4]:
tables = merge_historical_and_ergast_kaggle_tables(
    KAGGLE_ROOT,
    ERGAST_ROOT,
    ergast_years=ERGAST_YEARS,
    min_year=MIN_MASTER_YEAR,
    ergast_start_year=ERGAST_START_YEAR,
)

for name, df in tables.items():
  print(f"{name:22s} {len(df):>7,} rows  cols={len(df.columns)}")

drivers                    861 rows  cols=9
constructors               212 rows  cols=5
circuits                    77 rows  cols=9
races                      601 rows  cols=18
results                 27,238 rows  cols=18
qualifying              10,973 rows  cols=9
driver_standings        35,361 rows  cols=7
constructor_standings   13,631 rows  cols=7
constructor_results     12,865 rows  cols=5
sprint_results             480 rows  cols=16


C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\src\ergast_kaggle_adapter.py:575: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out[table] = pd.concat([hist, erg], ignore_index=True)


In [5]:
master = build_master_races(tables, min_year=MIN_MASTER_YEAR)
master["date"] = pd.to_datetime(master["date"], errors="coerce")
master = master.sort_values(["year", "round", "date"]).reset_index(drop=True)

print(f"Master shape: {master.shape}")
print(f"Years: {int(master['year'].min())} – {int(master['year'].max())}")
print(f"Rows per year (last 3):")
print(master.groupby("year").size().tail(3))

Master shape: (12837, 82)
Years: 1994 – 2025
Rows per year (last 3):
year
2023.0    440
2024.0    479
2025.0    479
dtype: int64


In [6]:
master.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")

Saved → C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\processed\master_races_ergast.csv


## Next steps

Run in order:
1. `04.2_augment_master_ergast.ipynb`
2. `04.3_feature_engineering_ergast.ipynb`
3. `04.4_feature_refinement_ergast.ipynb`
4. `04.5_missing_value_handling_ergast.ipynb`
5. `04.6_fastf1_feature_engineering_ergast.ipynb`